In [2]:
import mlflow
import pandas as pd
from sklearn import datasets
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import numpy as np
from sklearn import linear_model
mlflow.set_tracking_uri('sqlite:///mlflow.db')
mlflow.set_experiment('SCD')


/usr/local/python/3.12.1/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


<Experiment: artifact_location='mlflow-artifacts:/0', creation_time=1779782786199, experiment_id='0', last_update_time=1779786251550, lifecycle_stage='active', name='SCD', tags={}, trace_location=None, workspace='default'>

In [25]:
#Import data and convert the variable types to the correct format for modeling
scd = pd.read_excel('full_data.xlsx')
columns_to_clean = ['ACR','Creatinine','Alb/mg/l']
for col in columns_to_clean:
    if col in scd.columns:
        scd[col] = scd[col].astype(str).str.replace('<', '').str.replace('>', '')
scd['Creatinine'] = pd.to_numeric(scd['Creatinine'],errors = 'coerce')
scd['Alb/mg/l'] = pd.to_numeric(scd['Alb/mg/l'],errors = 'coerce')
scd['ACR'] = pd.to_numeric(scd['ACR'],errors = 'coerce')
# Assign HMOX1 genotype based on allele values
scd['HMOX1 GENOTYPE'] = np.where(
    (scd['HMOX1 ALLELE 1'] < 25) & (scd['HMOX1 ALLELE 2'] < 25),
    'SS',
    np.where(
        (scd['HMOX1 ALLELE 1'] > 25) & (scd['HMOX1 ALLELE 2'] < 25),
        'LS',
        np.where(
            (scd['HMOX1 ALLELE 1'] < 25) & (scd['HMOX1 ALLELE 2'] > 25),
            'SL',
            np.where(
                (scd['HMOX1 ALLELE 1'] > 25) & (scd['HMOX1 ALLELE 2'] > 25),
                'LL',
                None
            )
        )
    )
)
scd['HMOX1 GENOTYPE'].value_counts()
scd.info()





<class 'pandas.core.frame.DataFrame'>
RangeIndex: 266 entries, 0 to 265
Data columns (total 31 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   DNA NUMBER        266 non-null    object 
 1   APOL1 GENOTYPE    266 non-null    object 
 2   HMOX1 ALLELE 1    265 non-null    float64
 3   HMOX1 ALLELE 2    265 non-null    float64
 4   HMOX1 GENOTYPE    251 non-null    object 
 5   HMOX1_code        265 non-null    float64
 6   GENDER            266 non-null    object 
 7   AGE(Years)        266 non-null    int64  
 8   ETHNIC Group      266 non-null    object 
 9   BP                266 non-null    object 
 10  SBP               266 non-null    int64  
 11  DBP               266 non-null    int64  
 12  WEIGHT/Kg         266 non-null    float64
 13  HEIGHT/m          266 non-null    float64
 14  BMI (Kg/m2)       266 non-null    float64
 15  prot              266 non-null    object 
 16  GB/Nitr           266 non-null    object 
 1

In [38]:
scd1 =pd.get_dummies(scd, columns=['GENDER','APOL1 GENOTYPE','HMOX1 GENOTYPE','prot','GB/Nitr'])
scd1.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 266 entries, 0 to 265
Data columns (total 65 columns):
 #   Column                             Non-Null Count  Dtype  
---  ------                             --------------  -----  
 0   DNA NUMBER                         266 non-null    object 
 1   HMOX1 ALLELE 1                     265 non-null    float64
 2   HMOX1 ALLELE 2                     265 non-null    float64
 3   HMOX1_code                         265 non-null    float64
 4   AGE(Years)                         266 non-null    int64  
 5   ETHNIC Group                       266 non-null    object 
 6   BP                                 266 non-null    object 
 7   SBP                                266 non-null    int64  
 8   DBP                                266 non-null    int64  
 9   WEIGHT/Kg                          266 non-null    float64
 10  HEIGHT/m                           266 non-null    float64
 11  BMI (Kg/m2)                        266 non-null    float64

In [46]:
#Remove unneeded columns and columns with
#  clinical factors that are measured only on a few subjects
scd2= scd1.drop(columns=["BP","DNA NUMBER","ACR_code","ETHNIC Group","HMOX1_code","LDH","Fetal Hb","Bili totale","bili DIRECT","BILI INDIRECT"])
scd2 = scd2.dropna(axis=0, how='any')
X = scd2.drop(columns=["GFR"])
y = scd2['GFR']
X = X.dropna(axis=0, how='any')
X.info()
X_train_df, X_test_df, Y_train_df, Y_test_df = train_test_split(X, y, test_size=0.2, random_state=42)
X_train_df.info()
mlflow.sklearn.autolog()
model = linear_model.LinearRegression()
model.fit(X_train_df, Y_train_df)

<class 'pandas.core.frame.DataFrame'>
Index: 251 entries, 0 to 265
Data columns (total 54 columns):
 #   Column                             Non-Null Count  Dtype  
---  ------                             --------------  -----  
 0   HMOX1 ALLELE 1                     251 non-null    float64
 1   HMOX1 ALLELE 2                     251 non-null    float64
 2   AGE(Years)                         251 non-null    int64  
 3   SBP                                251 non-null    int64  
 4   DBP                                251 non-null    int64  
 5   WEIGHT/Kg                          251 non-null    float64
 6   HEIGHT/m                           251 non-null    float64
 7   BMI (Kg/m2)                        251 non-null    float64
 8   ACR                                251 non-null    float64
 9   Alb/mg/l                           251 non-null    float64
 10  Creatinine                         251 non-null    float64
 11  VOC                                251 non-null    int64  
 12 

2026/05/28 11:06:45 INFO mlflow.utils.autologging_utils: Created MLflow autologging run with ID '3328eea0235d4842a8a58539f5eed7be', which will track hyperparameters, performance metrics, model artifacts, and lineage information for the current sklearn workflow
2026/05/28 11:06:45 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/usr/local/python/3.12.1/lib/python3.12/site-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing V

,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies a different convergence criterion for the `lsqr` solver.`tol` is set as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. This parameter has no effect when fittingon dense data... versionadded:: 1.7",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary ` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False


In [ ]:
mlflow.start_run()
mlflow.set_tag('developer','Oluyomi')
mlflow.log_param('data-path', '908D4730.xlsx')